In [11]:
# Import shared loading/cleaning functions from utils.py
from utils import load_clean_data
from utils import get_cv_folds

import pandas as pd
import numpy as np

# Models we will train and compare
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import StackingRegressor

# For scaling numeric features (needed for Linear Regression and KNN)
from sklearn.preprocessing import StandardScaler

# For one-hot encoding the low-cardinality categorical columns
from sklearn.preprocessing import OneHotEncoder

# For CatBoost encoding the high-cardinality categorical columns
import category_encoders as ce

# For evaluation metrics
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

# For model interpretation: permutation importance and partial dependence
from sklearn.inspection import permutation_importance, PartialDependenceDisplay

# For SHAP values
import shap

import matplotlib.pyplot as plt

In [12]:
# Load the cleaned dataset
df = load_clean_data("data.csv")

# Generate the FIXED set of cross-validation folds
# Using the same random_state (set inside get_cv_folds) ensures these folds
# are identical to the folds used in every other encoder notebook
folds = get_cv_folds(df)

print(f"Loaded {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Generated {len(folds)} folds")

Loaded 79159 rows, 17 columns
Generated 10 folds


In [13]:
# The target variable we're predicting
y = df["advertised_price"]

# Drop the target from the feature set, everything else is a potential predictor
X = df.drop(columns=["advertised_price"])

# Numeric columns, used as-is in both configs
# NOTE: "engine" belongs here, not in the categorical lists below — it's
# treated as a numeric feature throughout the project (see utils.py, where
# missing engine values are filled with 0 rather than encoded as a category)
numeric_cols = ["mileage", "list_price", "reg_year", "doors", "engine",
                 "engine_missing", "vehicle_age_days", "vehicle_age_years"]

# ---- Config C: raw columns, split by cardinality ----
# Low-cardinality (< 50 unique values) -> One-Hot
ohe_cols = ["transmission", "body", "fuel", "plate_age_id"]

# High-cardinality (>= 50 unique values) -> CatBoost/Target encoding
cbe_cols = ["make", "model", "generation", "trim"]

# ---- Config D: same as C, but make/model/trim are replaced with engineered
# concatenated interaction columns (make_model, make_model_trim) instead of
# being encoded individually. generation stays as its own column.
cbe_cols_d = ["generation"]
interaction_cols = ["make_model", "make_model_trim"]

print(f"OHE columns: {len(ohe_cols)}")
print(f"CBE columns (Config C): {len(cbe_cols)}")
print(f"CBE columns (Config D, raw + interaction): {len(cbe_cols_d) + len(interaction_cols)}")

OHE columns: 4
CBE columns (Config C): 4
CBE columns (Config D, raw + interaction): 3


In [14]:
# Reset results dictionary, storing RMSE, MAE, and R² for each model,
# separately for Config C and Config D
model_names = ["Linear Regression", "KNN", "Decision Tree", "Random Forest", "Stacking"]

results = {
    "column_adaptive_c": {m: {"rmse": [], "mae": [], "r2": []} for m in model_names},
    "column_adaptive_d": {m: {"rmse": [], "mae": [], "r2": []} for m in model_names}
}

# Holds the LAST fold's fitted models and data, per config — used afterwards
# for feature importance, permutation importance, PDP, and SHAP. Keeping a
# single fold's snapshot (rather than all 10) matches how feature importance
# was already handled in every other encoder notebook.
last_fold_models = {"column_adaptive_c": {}, "column_adaptive_d": {}}
last_fold_data = {}

In [15]:
# Loop through each of the 10 fixed folds
for fold_num, (train_idx, test_idx) in enumerate(folds, start=1):

    # ---- Split data into this fold's train/test portions ----
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # ================= CONFIG C: raw columns, mixed encoding =================

    # ---- One-Hot Encode the low-cardinality categorical columns ----
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    ohe.fit(X_train[ohe_cols])  # fit on TRAINING data only, prevents leakage

    ohe_train = pd.DataFrame(
        ohe.transform(X_train[ohe_cols]),
        columns=ohe.get_feature_names_out(ohe_cols),
        index=X_train.index
    )
    ohe_test = pd.DataFrame(
        ohe.transform(X_test[ohe_cols]),
        columns=ohe.get_feature_names_out(ohe_cols),
        index=X_test.index
    )

    # ---- CatBoost Encode the high-cardinality categorical columns ----
    cbe = ce.CatBoostEncoder(cols=cbe_cols, random_state=42)
    cbe.fit(X_train[cbe_cols], y_train)  # fit on TRAINING data only
    cbe_train_c = cbe.transform(X_train[cbe_cols])
    cbe_test_c = cbe.transform(X_test[cbe_cols])

    X_train_c = pd.concat([X_train[numeric_cols], ohe_train, cbe_train_c], axis=1)
    X_test_c = pd.concat([X_test[numeric_cols], ohe_test, cbe_test_c], axis=1)

    scaler_c = StandardScaler()
    X_train_c_scaled = scaler_c.fit_transform(X_train_c)
    X_test_c_scaled = scaler_c.transform(X_test_c)

    # ================= CONFIG D: engineered interaction columns =================

    X_train_d = X_train.copy()
    X_test_d = X_test.copy()

    X_train_d["make_model"] = X_train["make"] + "_" + X_train["model"]
    X_train_d["make_model_trim"] = X_train["make"] + "_" + X_train["model"] + "_" + X_train["trim"]
    X_test_d["make_model"] = X_test["make"] + "_" + X_test["model"]
    X_test_d["make_model_trim"] = X_test["make"] + "_" + X_test["model"] + "_" + X_test["trim"]

    cbe_cols_d_all = cbe_cols_d + interaction_cols
    cbe_d = ce.CatBoostEncoder(cols=cbe_cols_d_all, random_state=42)
    cbe_d.fit(X_train_d[cbe_cols_d_all], y_train)
    cbe_train_d = cbe_d.transform(X_train_d[cbe_cols_d_all])
    cbe_test_d = cbe_d.transform(X_test_d[cbe_cols_d_all])

    # Reuses ohe_train / ohe_test from Config C — the low-cardinality columns
    # are identical between configs, so re-fitting would be redundant
    X_train_d_final = pd.concat([X_train[numeric_cols], ohe_train, cbe_train_d], axis=1)
    X_test_d_final = pd.concat([X_test[numeric_cols], ohe_test, cbe_test_d], axis=1)

    scaler_d = StandardScaler()
    X_train_d_scaled = scaler_d.fit_transform(X_train_d_final)
    X_test_d_scaled = scaler_d.transform(X_test_d_final)

    # ================= Train and evaluate all 5 models, for both configs =================

    for cfg_name, X_train_final, X_test_final, X_train_scaled, X_test_scaled in [
        ("column_adaptive_c", X_train_c, X_test_c, X_train_c_scaled, X_test_c_scaled),
        ("column_adaptive_d", X_train_d_final, X_test_d_final, X_train_d_scaled, X_test_d_scaled),
    ]:
        # ---- Define the 5 models ----
        linear_model = LinearRegression()
        knn_model = KNeighborsRegressor()
        tree_model = DecisionTreeRegressor(random_state=42)
        rf_model = RandomForestRegressor(random_state=42, n_estimators=50)

        stacking_model = StackingRegressor(
            estimators=[
                ("linear", LinearRegression()),
                ("knn", KNeighborsRegressor()),
                ("tree", DecisionTreeRegressor(random_state=42)),
                ("rf", RandomForestRegressor(random_state=42, n_estimators=50))
            ],
            final_estimator=LinearRegression(),
            cv=3,
        )

        models_to_run = [
            ("Linear Regression", linear_model, X_train_scaled, X_test_scaled),
            ("KNN", knn_model, X_train_scaled, X_test_scaled),
            ("Decision Tree", tree_model, X_train_final, X_test_final),
            ("Random Forest", rf_model, X_train_final, X_test_final),
            ("Stacking", stacking_model, X_train_final, X_test_final),
        ]

        for model_name, model, Xtr, Xte in models_to_run:
            model.fit(Xtr, y_train)
            preds = model.predict(Xte)

            results[cfg_name][model_name]["rmse"].append(root_mean_squared_error(y_test, preds))
            results[cfg_name][model_name]["mae"].append(mean_absolute_error(y_test, preds))
            results[cfg_name][model_name]["r2"].append(r2_score(y_test, preds))

            # Keep the LAST fold's fitted model for downstream interpretation
            last_fold_models[cfg_name][model_name] = model

    # Keep the LAST fold's data (both configs) for downstream interpretation
    last_fold_data = {
        "column_adaptive_c": {"X_train": X_train_c, "X_test": X_test_c, "y_train": y_train, "y_test": y_test},
        "column_adaptive_d": {"X_train": X_train_d_final, "X_test": X_test_d_final, "y_train": y_train, "y_test": y_test},
    }

    # Print progress so we can see the loop working fold by fold
    print(f"Fold {fold_num}/{len(folds)} complete")

print("\nAll folds complete.")

KeyboardInterrupt: 

In [8]:
# Build a summary table: mean and std of RMSE, MAE, and R² across the 10 folds,
# per model, per config
for cfg_name, cfg_results in results.items():
    print(f"\n{cfg_name}")
    summary_rows = []
    for model_name, metrics in cfg_results.items():
        summary_rows.append({
            "model": model_name,
            "mean_rmse": np.mean(metrics["rmse"]),
            "std_rmse": np.std(metrics["rmse"]),
            "mean_mae": np.mean(metrics["mae"]),
            "std_mae": np.std(metrics["mae"]),
            "mean_r2": np.mean(metrics["r2"]),
            "std_r2": np.std(metrics["r2"])
        })
    summary = pd.DataFrame(summary_rows)
    print(summary)


column_adaptive_c
               model    mean_rmse    std_rmse     mean_mae    std_mae  \
0  Linear Regression  4133.197653  125.278765  2562.698669  36.572065   
1                KNN  3298.735970  159.257538  1704.938657  28.378668   
2      Decision Tree  2710.191590  179.199415  1495.306608  25.107326   
3      Random Forest  1945.058357  156.838625  1068.131941  19.644201   
4           Stacking  1951.021178  148.813943  1089.353525  19.601862   

    mean_r2    std_r2  
0  0.913549  0.003793  
1  0.944902  0.004285  
2  0.962777  0.004082  
3  0.980797  0.002755  
4  0.980689  0.002585  

column_adaptive_d
               model    mean_rmse    std_rmse     mean_mae    std_mae  \
0  Linear Regression  4031.779256  143.009022  2464.200554  37.455453   
1                KNN  3238.337961  180.169769  1658.287537  34.674282   
2      Decision Tree  2832.418478  183.524855  1526.599262  36.785516   
3      Random Forest  2076.178480  165.893497  1097.650994  27.850499   
4           St

In [9]:
# Flatten results into a long-format DataFrame: one row per config per model
# per fold per metric (now including fit time and peak memory)
rows = []
for cfg_name, cfg_results in results.items():
    for model_name, metrics in cfg_results.items():
        for fold_i in range(len(folds)):
            rows.append({
                "encoder": cfg_name,
                "model": model_name,
                "fold": fold_i + 1,
                "rmse": metrics["rmse"][fold_i],
                "mae": metrics["mae"][fold_i],
                "r2": metrics["r2"][fold_i],
            })

results_df = pd.DataFrame(rows)
results_df.to_csv("column_adaptive_scores.csv", index=False)

print("Saved results to column_adaptive_scores.csv")

Saved results to column_adaptive_scores.csv


In [10]:
# Capture built-in feature importance from Random Forest for BOTH configs,
# using the model and data from the LAST fold — a reasonable single snapshot,
# consistent with how this was done in every other encoder notebook
rf_c = last_fold_models["column_adaptive_c"]["Random Forest"]
rf_d = last_fold_models["column_adaptive_d"]["Random Forest"]

fi_c = pd.DataFrame({
    "feature": last_fold_data["column_adaptive_c"]["X_train"].columns,
    "importance": rf_c.feature_importances_
}).sort_values("importance", ascending=False)
fi_c["encoder"] = "column_adaptive_c"

fi_d = pd.DataFrame({
    "feature": last_fold_data["column_adaptive_d"]["X_train"].columns,
    "importance": rf_d.feature_importances_
}).sort_values("importance", ascending=False)
fi_d["encoder"] = "column_adaptive_d"

feature_importance = pd.concat([fi_c, fi_d], ignore_index=True)
feature_importance.to_csv("column_adaptive_feature_importance.csv", index=False)

print("Config C — top 10 features (built-in RF importance)")
print(fi_c.head(10).to_string(index=False))

print("\nConfig D — top 10 features (built-in RF importance)")
print(fi_d.head(10).to_string(index=False))

KeyError: 'Random Forest'